In [1]:
import pandas as pd
from pathlib import Path

raw_dir = Path("../data/raw")

df21 = pd.read_excel(raw_dir / "fdinspi_2122.xlsx")
df22 = pd.read_excel(raw_dir / "fdinspi_2223.xlsx")
df23 = pd.read_excel(raw_dir / "fdinspi_2324.xlsx")
df24 = pd.read_excel(raw_dir / "fdinspi_2425.xlsx")
df25 = pd.read_excel(raw_dir / "fdinspi_2526.xlsx")

df21["fiscal_year"] = "2021-22"
df22["fiscal_year"] = "2022-23"
df23["fiscal_year"] = "2023-24"
df24["fiscal_year"] = "2024-25"
df25["fiscal_year"] = "2025-26"

In [2]:
print("FY24-25 columns:", len(df24.columns))
print("FY25-26 columns:", len(df25.columns))

FY24-25 columns: 84
FY25-26 columns: 84


In [3]:
rename_columns = dict(zip(df24.columns, df25.columns))

df21 = df21.rename(columns=rename_columns)
df22 = df22.rename(columns=rename_columns)
df23 = df23.rename(columns=rename_columns)
df24 = df24.rename(columns=rename_columns)

In [4]:
print(df21.columns.equals(df25.columns))
print(df22.columns.equals(df25.columns))
print(df23.columns.equals(df25.columns))
print(df24.columns.equals(df25.columns))

True
True
True
True


In [5]:
df = pd.concat(
    [df21, df22, df23, df24, df25],
    ignore_index=True
)

In [6]:
unnamed_columns = [
    column for column in df.columns
    if str(column).startswith("Unnamed")
]

df = df.drop(columns=unnamed_columns)

In [7]:
df["Inspection Date"] = pd.to_datetime(
    df["Inspection Date"],
    errors="coerce"
)

In [8]:
print(df.shape)
df.head()

(681641, 83)


,District,County Number,County Name,License Type Code,License Number,Business (DBA-Does Business As) Name,Location Address,Location City,Location Zip Code,Inspection Number,...,Violation 52,Violation 53,Violation 54,Violation 55,Violation 56,Violation 57,Violation 58,License ID,Inspection Visit ID,fiscal_year
0,D3,62,Pinellas,2010,6215291,GRANTS CRABS SEAFOOD AND GRILLE,13030 STARKEY RD UNIT 3,LARGO,33773,3108311,...,0,0,0,0,0,0,0,0,6275376,2021-22
1,D3,70,Sumter,2010,7000657,FIESTA GRANDE MEXICAN GRILL,3647 KIESSEL RD,THE VILLAGES,32163,3177676,...,0,0,0,0,0,0,0,0,7807573,2021-22
2,D2,16,Broward,2010,1616635,CHECKERS #439,2701 W BROWARD BLVD,FORT LAUDERDALE,33312-1245,3155324,...,0,0,0,0,0,0,0,0,2145237,2021-22
3,D3,62,Pinellas,2014,6252194,GO STUFF URSELF,1540 70 ST N,ST. PETERSBURG,33710,1238275,...,0,0,0,0,0,0,0,0,6970227,2021-22
4,D2,53,Martin,2014,5350334,JAKE'S ATOMIC CAFE,3620 SE DIXIE HWY,STUART,34997-5247,1238670,...,0,0,1,0,0,0,0,0,7856355,2021-22


In [9]:
df["fiscal_year"].value_counts().sort_index()

fiscal_year
2021-22    132394
2022-23    128529
2023-24    138045
2024-25    140470
2025-26    142203
Name: count, dtype: int64

In [14]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

In [16]:
[column for column in df.columns if "License" in column]

[' License Type Code', ' License Number', 'License ID']

In [17]:
df[[
    "License ID",
    "Business (DBA-Does Business As) Name",
    "Location Address"
]].head(20)

,License ID,Business (DBA-Does Business As) Name,Location Address
0,0,GRANTS CRABS SEAFOOD AND GRILLE,13030 STARKEY RD UNIT 3
1,0,FIESTA GRANDE MEXICAN GRILL,3647 KIESSEL RD
2,0,CHECKERS #439,2701 W BROWARD BLVD
3,0,GO STUFF URSELF,1540 70 ST N
4,0,JAKE'S ATOMIC CAFE,3620 SE DIXIE HWY
5,0,WESTIN CAFE,400 CORPORATE DR
6,0,PANDA EXPRESS 2482,4260 S US HWY 17-92
7,0,KELSEY'S PIZZA,6811 N US HWY 1
8,0,MI JALISCO,7700 NORTH WICKHAM ROAD STE 101
9,0,TACO BELL 26458,2224 NW 13 ST


In [18]:
df["License ID"].value_counts().head(20)

License ID
0    681640
1         1
Name: count, dtype: int64

In [11]:
df.to_csv(
    processed_dir / "florida_restaurant_inspections.csv.gz",
    index=False,
    compression="gzip"
)

In [12]:
(processed_dir / "florida_restaurant_inspections.csv.gz").exists()

True

In [13]:
df = pd.read_csv(
    "../data/processed/florida_restaurant_inspections.csv.gz",
    parse_dates=["Inspection Date"],
    low_memory=False
)